In [7]:
import os

os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

In [ ]:
from pathlib import Path

import nibabel as nib

In [ ]:
import nitransforms

coregistration_transform_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-boldref_to-T1w_mode-image_desc-coreg_xfm.txt")
coregistration_transform = nitransforms.linear.load(coregistration_transform_path)

derivatives_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5")

boldref = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_desc-hmc_boldref.nii.gz"
)

In [43]:
t1w_image_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5/anat/sub-NDARINVXP963GZ5_run-01_desc-preproc_T1w.nii.gz")
t1w_image = nib.nifti1.load(t1w_image_path) # pyright: ignore[reportAttributeAccessIssue]

t1w_ornt = nib.io_orientation(t1w_image.affine)
boldref_ornt = nib.io_orientation(boldref.affine)
ornt_trans = nib.orientations.ornt_transform(t1w_ornt, boldref_ornt)

t1w_image = t1w_image.as_reoriented(ornt_trans)

t1w_data = t1w_image.get_fdata()
t1w_data.shape

(144, 192, 192)

In [44]:
nib.aff2axcodes(t1w_image.affine)

('L', 'A', 'S')

In [45]:
data = boldref.get_fdata()
data.shape

(104, 104, 72)

In [52]:
from collections.abc import MutableSequence
from typing import Any

import numpy as np
import pyvista as pv

plot_slices = (
    ("x", "yz"),
    ("y", "xz"),
    ("z", "xy"),
)
plot_voxel = np.array([52, 48, 21])

window_size = [3840, 2160]
aspect = (window_size[0] / len(plot_slices)) / window_size[1]

fixed_cameras: MutableSequence[Any] = [None, None, None]
volume_center = None
scales: MutableSequence[float | None] = [None, None, None]

grid = pv.ImageData(
    dimensions=np.array(data.shape[:3]) + 1,
    spacing=(1.0, 1.0, 1.0),
    origin=(-0.5, -0.5, -0.5),
)

t1w_grid = pv.ImageData(
    dimensions=np.array(t1w_data.shape[:3]) + 1,
    spacing=(1.0, 1.0, 1.0),
    origin=(-0.5, -0.5, -0.5),
)

volume_center = plot_voxel

grid.cell_data["current"] = data.flatten(order="F")
t1w_grid.cell_data["current"] = t1w_data.flatten(order="F")

transformed_grid = grid.cast_to_structured_grid()
transformed_t1w_grid = t1w_grid.cast_to_structured_grid()

transformed_t1w_grid = transformed_t1w_grid.transform(
    np.linalg.inv(boldref.affine) @ coregistration_transform.matrix @ t1w_image.affine,
    inplace=False,
)
# transformed_grid = transformed_t1w_grid

plotter = pv.Plotter(
    shape=(1, len(plot_slices)),
    off_screen=True,
    window_size=window_size,
    border=False,
)
for i, (normal, camera_position) in enumerate(plot_slices):
    plotter.subplot(0, i)

    plotter.set_background("black")  # pyright: ignore[reportArgumentType]

    intersection = transformed_grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
    t1w_intersection = transformed_t1w_grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
    plotter.add_mesh(
        t1w_intersection,
        scalars="current",
        cmap="gray",
        lighting=False,
        show_edges=False,
        show_scalar_bar=False,
    )
    plotter.enable_anti_aliasing("ssaa")

    if scales[i] is None:
        bounds = intersection.bounds
        sizes = [
            bounds.x_max - bounds.x_min,
            bounds.y_max - bounds.y_min,
            bounds.z_max - bounds.z_min,
        ]
        sizes.pop(i)
        scale = (
            max(sizes[1], sizes[0] / aspect)
            * 0.5
            * {"x": 0.9, "y": 0.8, "z": 0.7}[normal]
        )
        scales[i] = scale
    if fixed_cameras[i] is None:
        plotter.camera_position = camera_position  # pyright: ignore[reportAttributeAccessIssue]
        fixed_cameras[i] = [tuple(v) for v in plotter.camera_position]
    plotter.camera_position = fixed_cameras[i]

    plotter.enable_parallel_projection()  # pyright: ignore[reportCallIssue]
    plotter.camera.parallel_scale = scales[i]
    plotter.reset_camera_clipping_range()

plotter.screenshot("anatomical.png")
plotter.close()

In [53]:
from collections.abc import MutableSequence
from typing import Any

import numpy as np
import pyvista as pv

plot_slices = (
    ("x", "yz"),
    ("y", "xz"),
    ("z", "xy"),
)
plot_voxel = np.array([52, 48, 21])

window_size = [3840, 2160]
aspect = (window_size[0] / len(plot_slices)) / window_size[1]

fixed_cameras: MutableSequence[Any] = [None, None, None]
volume_center = None
scales: MutableSequence[float | None] = [None, None, None]

grid = pv.ImageData(
    dimensions=np.array(data.shape[:3]) + 1,
    spacing=(1.0, 1.0, 1.0),
    origin=(-0.5, -0.5, -0.5),
)

t1w_grid = pv.ImageData(
    dimensions=np.array(t1w_data.shape[:3]) + 1,
    spacing=(1.0, 1.0, 1.0),
    origin=(-0.5, -0.5, -0.5),
)

volume_center = plot_voxel

grid.cell_data["current"] = data.flatten(order="F")
t1w_grid.cell_data["current"] = t1w_data.flatten(order="F")

transformed_grid = grid.cast_to_structured_grid()
transformed_t1w_grid = t1w_grid.cast_to_structured_grid()

transformed_t1w_grid = transformed_t1w_grid.transform(
    np.linalg.inv(boldref.affine) @ coregistration_transform.matrix @ t1w_image.affine,
    inplace=False,
)
# transformed_grid = transformed_t1w_grid

plotter = pv.Plotter(
    shape=(1, len(plot_slices)),
    off_screen=True,
    window_size=window_size,
    border=False,
)
for i, (normal, camera_position) in enumerate(plot_slices):
    plotter.subplot(0, i)

    plotter.set_background("black")  # pyright: ignore[reportArgumentType]

    intersection = transformed_grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
    t1w_intersection = transformed_t1w_grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
    plotter.add_mesh(
        intersection,
        scalars="current",
        cmap="gray",
        lighting=False,
        show_edges=False,
        show_scalar_bar=False,
    )
    plotter.enable_anti_aliasing("ssaa")

    if scales[i] is None:
        bounds = intersection.bounds
        sizes = [
            bounds.x_max - bounds.x_min,
            bounds.y_max - bounds.y_min,
            bounds.z_max - bounds.z_min,
        ]
        sizes.pop(i)
        scale = (
            max(sizes[1], sizes[0] / aspect)
            * 0.5
            * {"x": 0.9, "y": 0.8, "z": 0.7}[normal]
        )
        scales[i] = scale
    if fixed_cameras[i] is None:
        plotter.camera_position = camera_position  # pyright: ignore[reportAttributeAccessIssue]
        fixed_cameras[i] = [tuple(v) for v in plotter.camera_position]
    plotter.camera_position = fixed_cameras[i]

    plotter.enable_parallel_projection()  # pyright: ignore[reportCallIssue]
    plotter.camera.parallel_scale = scales[i]
    plotter.reset_camera_clipping_range()

plotter.screenshot("functional.png")
plotter.close()